<img src="https://hilpisch.com/tpq_logo_bic.png" width="320" align="right">

# Reinforcement Learning for Dynamic Asset Allocation

## A2C Actor-Critic Agent Workbench

Dr. Yves J. Hilpisch | https://tpq.io

This notebook presents the A2C Beta-policy actor-critic experiment as a compact result story. It starts with the configuration, prepares the runtime context, evaluates aggregate baselines, trains the A2C policy, compares aggregate test-slice behaviour, and closes with one selected-slice inspection.

## Configuration

The experiment profile is read from the reusable YAML configuration used by the A2C actor-critic run.

The selected profile keeps the A2C-policy parameters in the same configuration family as the other workbenches.

## Available Symbols

`AAPL, NVDA, JPM, SPY, GLD, TLT, EURUSD, BTC-USD`

## Runtime Setup

The runtime setup imports the notebook helpers and resolves the project root. These helpers keep execution logic in the package and leave this notebook focused on presentation.

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [ ]:
%config InlineBackend.figure_format = 'svg'

The following import block loads the functions used for configuration, training, comparison, and visual inspection.

In [ ]:
from rl4am.notebooks.a2c_workbench import (
    apply_a2c_training_overrides,
    build_comparison_artifacts,
    build_a2c_config_overview_frame,
    build_a2c_training_summary_frame,
    build_sweep_winner_frame,
    build_publication_figure,
    build_runtime_overview_frame,
    build_sampling_overview_frame,
    build_selected_slice_story,
    compact_metric_frame,
    configure_notebook_display,
    load_a2c_workbench_config,
    plot_a2c_training_history,
    plot_visual_diagnostics,
    prepare_a2c_context,
    resolve_project_root,
    run_baselines,
    run_a2c_training,
)

configure_notebook_display()
ROOT = resolve_project_root(Path.cwd())

The runtime constants select the reusable configuration file, which baseline is used in comparisons, which test slice is inspected at the end, and whether the notebook uses a separate evaluation draw.

In [ ]:
CONFIG_PATH = (ROOT / 'configs' / 'a2c_best.yml') if (ROOT / 'configs' / 'a2c_best.yml').exists() else (ROOT / 'configs' / 'a2c_ref.yml')
TRAIN_SLICES_OVERRIDE = None
RUNS_PER_TRAIN_SLICE_OVERRIDE = None
DEFAULT_BASELINE_NAME = 'grid_best'
DEFAULT_TEST_SLICE = 0
EVAL_SAMPLING_SEED = 4444
EVAL_TEST_SLICES_OVERRIDE = 10
TRAINING_DEVICE = 'cpu'

The context resolves the YAML configuration, loads market data, samples training slices, optionally samples a separate evaluation draw, and creates the run directories.

The optional training-count overrides distinguish the number of sampled training slices from the number of passes over each slice.

In [ ]:
config, config_notes = load_a2c_workbench_config(config_path=CONFIG_PATH)
config = apply_a2c_training_overrides(
    config,
    train_slices=TRAIN_SLICES_OVERRIDE,
    runs_per_train_slice=RUNS_PER_TRAIN_SLICE_OVERRIDE,
)
context = prepare_a2c_context(
    root=ROOT,
    config_path=CONFIG_PATH,
    config=config,
    config_notes=config_notes,
    selected_test_slice=DEFAULT_TEST_SLICE,
    evaluation_seed=EVAL_SAMPLING_SEED,
    evaluation_test_slices=EVAL_TEST_SLICES_OVERRIDE,
    baseline_name=DEFAULT_BASELINE_NAME,
)

The configuration overview summarises the market and A2C policy assumptions used by the run.

In [ ]:
display(build_a2c_config_overview_frame(context))

The runtime overview records where the run artefacts and comparison outputs are written.

In [ ]:
display(
    build_runtime_overview_frame(context).query(
        "item != 'selected_test_slice'"
    )
)

The sampling overview summarises the data span and the train/test slicing plan.

In [ ]:
display(build_sampling_overview_frame(context))

If the configuration loader adjusted any notebook-facing fields, the notes below record those adjustments.

In [ ]:
if context.config_notes:
    display(Markdown("\n".join(
        f"- {note}" for note in context.config_notes
    )))

## Aggregate Baselines

The deterministic baselines are calibrated on each training slice, averaged into one fixed weight per strategy, and then evaluated on the full test-slice set. This section deliberately stays aggregate-only; individual slice details are reserved for the final inspection section.

In [ ]:
baselines = run_baselines(context)

The baseline setup table records the deterministic strategy parameters used in the comparison.

In [ ]:
display(baselines.baseline_setup.T)

The baseline summary aggregates performance across the sampled test slices.

In [ ]:
display(baselines.baseline_slice_summary.set_index(['strategy', 'stat']))

The diagnostic note below appears only when the baseline weights collapse to a common boundary or value.

In [ ]:
if baselines.boundary_note:
    display(Markdown(baselines.boundary_note))

## A2C Actor-Critic Training

The Beta-policy actor-critic is trained on the sampled training slices and then evaluated with its deterministic mean policy on the complete test-slice set.

In [ ]:
a2c = run_a2c_training(context, device=TRAINING_DEVICE)

The training summary compares the first and final update diagnostics.

In [ ]:
display(build_a2c_training_summary_frame(a2c))

The final rows of the training history show the latest reward and loss diagnostics.

In [ ]:
display(a2c.history_frame.tail())

The training-history plot visualises reward, actor-critic loss, and policy entropy over updates.

In [ ]:
plot_a2c_training_history(a2c.history_frame)

The policy summary aggregates the mean A2C policy performance across all test slices.

In [ ]:
display(a2c.slice_summary.set_index(['strategy', 'stat']))

The saved run path identifies the canonical result directory used by the comparison layer.

In [ ]:
display(Markdown(f'Saved A2C run to `{context.policy_dir}`.'))

## Aggregate Comparison

The comparison artefacts reload saved result files and compare the A2C policy with the selected baseline across the full test-slice set.

In [ ]:
comparison = build_comparison_artifacts(context, baselines, a2c)

The following table summarises metric differences between the policy and both benchmark references across all test slices.

In [ ]:
display(comparison.diff_summary)

The win-rate table reports how often the policy outperforms each benchmark by metric across the test-slice set.

In [ ]:
print(comparison.win_summary)

The best-slice table ranks the strongest A2C-policy test slices by terminal equity and annualised Sharpe.

In [ ]:
display(comparison.best_test_slices)

The sweep winner table lists the strongest successful A2C sweep runs when sweep artifacts are available.

In [ ]:
display(build_sweep_winner_frame(ROOT, sweep_dir='runs/a2c_sweep').T)

## Selected Slice Inspection

This final section selects one test slice for detailed inspection. Change `INSPECT_TEST_SLICE` or `INSPECT_BASELINE_NAME` to inspect another saved slice without rerunning training.

In [ ]:
INSPECT_TEST_SLICE = 6
INSPECT_BASELINE_NAME = 'grid_best'

The inspection context points the comparison helpers at the chosen test slice and baseline.

In [ ]:
inspection_context = prepare_a2c_context(
    root=ROOT,
    config_path=CONFIG_PATH,
    config=config,
    config_notes=config_notes,
    evaluation_seed=EVAL_SAMPLING_SEED,
    evaluation_test_slices=EVAL_TEST_SLICES_OVERRIDE,
    selected_test_slice=INSPECT_TEST_SLICE,
    baseline_name=INSPECT_BASELINE_NAME,
)
inspection = build_comparison_artifacts(
    inspection_context,
    baselines,
    a2c,
)

The selected-slice manifest identifies the evaluation window used for the detailed inspection.

In [ ]:
selected_slice = inspection_context.eval_slices[
    inspection_context.selected_test_slice
]
display({
    'slice_id': selected_slice.slice_id,
    'split': selected_slice.split,
    'start_date': selected_slice.start_date,
    'end_date': selected_slice.end_date,
    'rows': int(selected_slice.returns.shape[0]),
})

The selected-slice story table compares the policy and baseline on the main metrics for the chosen window.

In [ ]:
display(build_selected_slice_story(
    inspection,
    inspection_context.baseline_name,
))

The compact metric table provides the same selected-slice comparison in the standard metric layout.

In [ ]:
display(compact_metric_frame(inspection.selected_metrics))

The path diagnostics show net equity, turnover, and risky allocation for the selected slice.

In [ ]:
plot_visual_diagnostics(inspection)

The report figure displays the saved equity comparison for the selected slice.

In [ ]:
inspection_figure_path = build_publication_figure(
    inspection,
    inspection_context,
)

The output path below records where the selected-slice comparison assets are stored.

In [ ]:
display(Markdown(
    f'Comparison assets written to `{inspection_context.selected_report_dir}`.'
))

The saved figure is shown below for direct visual inspection.

In [ ]:
display(Image(filename=str(inspection_figure_path), width=760))

<img src="https://hilpisch.com/tpq_logo_bic.png" width="320" align="right">